# 原始模型的处理方式

In [2]:
import pickle
import os
import json
from pathlib import Path
import torch
from torch.utils.data import DataLoader

from multi_fusion_model.this_work import (
    Config as ThisWorkConfig,
    fit_experiment as fit_this_work_experiment,
)

pkl_path = "./datasets/附件2-数据集特征文件/aligned_50.pkl"

with open(pkl_path, "rb") as f:
    data = pickle.load(f)

if "data" not in globals():
    raise RuntimeError("请先运行数据加载代码，得到 data。")
if "MODULE_DIR" not in globals():
    MODULE_DIR = str(Path.cwd() / "results" / "second_question")
if "mask_overrides" not in globals():
    mask_overrides = None

In [3]:
this_work_config = ThisWorkConfig(
    task="multitask",
    classification_loss_weight=1.0,
    regression_loss_weight=1.0,
    selection_metric="balanced",
    output_root=os.path.join(MODULE_DIR, "this_work"),
)

this_work_experiment = fit_this_work_experiment(
    data,
    this_work_config,
    mask_overrides=globals().get("mask_overrides"),
)

print("验证结果:", this_work_experiment.valid_metrics)
print("测试结果:", this_work_experiment.test_metrics)
print("输出目录:", this_work_experiment.run_dir)

ThisWork multitask | device=cuda | classes=[0.0, 1.0, 2.0] | output=/data_disk/disk_1/hl/workspace/huawei_cup/MultiModel-Sentiment/results/second_question/this_work/20260925_150303_247629
CE weight=1.0; MAE weight=1.0; selection=balanced; fixed MAE scale=0.835444
warmup 01 | reconstruction=1.8263
warmup 02 | reconstruction=1.3242
warmup 03 | reconstruction=1.1819
epoch 01 | CE=0.9532 | MAE=0.7724 | valid Macro-F1=0.5337 | valid MAE=0.6550
epoch 02 | CE=0.7920 | MAE=0.6480 | valid Macro-F1=0.5530 | valid MAE=0.5880
epoch 03 | CE=0.7303 | MAE=0.5847 | valid Macro-F1=0.5449 | valid MAE=0.5950
epoch 04 | CE=0.6796 | MAE=0.5446 | valid Macro-F1=0.6000 | valid MAE=0.5842
epoch 05 | CE=0.6267 | MAE=0.5067 | valid Macro-F1=0.5834 | valid MAE=0.5905
epoch 06 | CE=0.5790 | MAE=0.4875 | valid Macro-F1=0.5953 | valid MAE=0.6284
epoch 07 | CE=0.5078 | MAE=0.4528 | valid Macro-F1=0.5914 | valid MAE=0.5839
epoch 08 | CE=0.4367 | MAE=0.4256 | valid Macro-F1=0.5669 | valid MAE=0.6138
epoch 09 | CE=0.40

# 对样本进行干扰，参考附件3的方式

In [1]:
import pickle

In [15]:
file_3_pkl_path = "/data_disk/disk_1/hl/workspace/huawei_cup/MultiModel-Sentiment/datasets/附件3-模态缺失特征样本/对齐版本/附件3_01.pkl"

with open(file_3_pkl_path, "rb") as f:
    file_3_data = pickle.load(f)

file_3_data

{'test': {'text_bert': array([[[1.010e+02, 2.021e+03, 2.048e+03, 2.041e+03, 1.997e+03,
           2.274e+03, 3.340e+03, 1.020e+02, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00],
          [1.000e+00, 1.000e+00, 1.000e+00, 1.000e+00, 1.000e+00,
           1.000e+00, 1.000e+00, 1.000e+00, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
           0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,

In [17]:
test_data = file_3_data['test']
test_data.keys()

dict_keys(['text_bert', 'vision', 'audio'])

In [35]:
print(test_data['text_bert'].shape)
print(test_data['vision'].shape)
print(test_data['audio'].shape)

(1, 3, 50)
(1, 50, 35)
(1, 50, 74)


In [29]:
print(test_data['text_bert'][:,0,:])
print(test_data['text_bert'][:,1,:])
print(test_data['text_bert'][:,2,:])

[[ 101. 2021. 2048. 2041. 1997. 2274. 3340.  102.    0.    0.    0.    0.
     0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
     0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
     0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
     0.    0.]]
[[1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]]
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]]


In [ ]:
print(test_data['vision'][1])


[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
